# Hard24 SFSA: history-conditioned comparators, v2

This notebook adds **DynamicDeepHit** (longitudinal adaptation), **SurvTransformer** (SurvTRACE history-feature adaptation with original PCH likelihood) and **SurvTransformer-Discrete** (discrete-step sensitivity variant). Earlier models are now **DynamicDeepHIt-Lite** and **SurvTransformer-Lite**.

Both new architectures receive every observed state through the prediction landmark, never future states. At baseline they receive x with no observed state. The duration-risk change at t=20 and hard Sicker/Sicker-to-Death rule from t=24 are unchanged. Model predictions are not externally forced to obey the rule.

The 11 learned models are compared with all seven Glazier sampling methods and the true-generator IID sampler. Full mode uses M=10,000 and five sampling seeds. Read **HISTORY_COMPARATORS.md** for exact replication choices and limitations. These are independent SFSA adaptations, not reproductions of clinical-study hyperparameter searches.

Open locally with gpu_env, or upload this notebook and **Glazier_Weibull_HardRule24_History_v2.zip** to Colab. Start with smoke mode. Full new training remains pending.

In [ ]:
import os, sys, datetime
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
if not hasattr(datetime,"UTC"): datetime.UTC=datetime.timezone.utc
IN_COLAB="google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if IN_COLAB:
    import subprocess
    subprocess.check_call([sys.executable,"-m","pip","-q","install","lifelines==0.30.3","matplotlib","pandas"])
from pathlib import Path
import zipfile
BUNDLE=Path.cwd()  # adjust for a custom extraction directory
candidates=[BUNDLE,BUNDLE/"Glazier_Weibull_HardRule24",BUNDLE/"outputs/Glazier_Weibull_HardRule24"]
found=next((p for p in candidates if (p/"hard_rule_workflow.py").exists()),None)
if found is None and IN_COLAB:
    from google.colab import files
    print("Upload Glazier_Weibull_HardRule24_History_v2.zip")
    uploaded=files.upload();archive=next((Path(k) for k in uploaded if k.endswith('.zip')),None)
    if archive is None: raise FileNotFoundError("Bundle ZIP required")
    dest=Path('/content/glazier_hard24').resolve();dest.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        if any(not (dest/e.filename).resolve().is_relative_to(dest) for e in z.infolist()):raise ValueError('Unsafe ZIP path')
        z.extractall(dest)
    found=dest/'Glazier_Weibull_HardRule24'
if found is None:raise FileNotFoundError('Extract the bundle and set BUNDLE to its folder')
BUNDLE=found.resolve()
for parent in [BUNDLE,*BUNDLE.parents]:
    support=parent/'work/gpu_support'
    if support.exists():sys.path.insert(0,str(support));break
sys.path[:0]=[str(BUNDLE),str(BUNDLE/'source')]
import torch,numpy as np,pandas as pd
from hard_rule_workflow import train,evaluate,plot
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(4)
print('Device:',torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU (full mode will be slow)')

from extension_workflow import ALL_SM_LABELS
print("Models:", ALL_SM_LABELS)

## Check models, rule and samplers
New checks cover hand-calculated losses, equal-elapsed-time ranking, tail mass, future-suffix invariance, observed-history sensitivity, checkpoint restoration and old-label protection. The existing checks cover the generator and all seven samplers. These are correctness checks, not performance experiments.

In [ ]:
import unittest, test_history_models
checks=unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromModule(test_history_models))
if not checks.wasSuccessful(): raise RuntimeError('History-model checks failed')
from test_rule import run_tests as check_rule
from test_samplers import run_tests as check_samplers
rule_checks=check_rule()
sampler_checks=check_samplers()

## Budget, persistent output and compatible checkpoint reuse
Full mode keeps 50,000 patients, the original 80/10/10 split, at most 200 joint-training epochs, patience 15 and landmarks 0,20,23,24. Every eligible training prefix is used, increasing training work relative to the Lite models. Glazier uses M=10,000; direct models use no rollouts.

Set **REUSE_FROM** to a completed hard24 results folder or its checkpoints subfolder to copy its cohort and unchanged models to a fresh output directory. Generator hashes, cohort size, seed and rule must match. Old model labels map to Lite. The original archive is preserved. For Colab, separately upload/extract that old results folder or mount it from Drive. Leave None to train everything anew.

Enable Drive for persistent Colab output. Change RUN_TAG whenever changing code/settings and restart the kernel after source edits.

In [ ]:
MODE="smoke"  # change to "full" for performance experiments
USE_DRIVE=False
REUSE_FROM=None  # path to compatible old hard24 results, or its checkpoints subfolder
RUN_TAG=MODE+'_hard24_history_v2'
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_PARENT=Path('/content/drive/MyDrive/Glazier_HardRule24')
else: OUTPUT_PARENT=BUNDLE/'results'
if MODE=='full':
    N=50000; EPOCHS=200; PATIENCE=15; M=10000; N_EVAL=300
    SEEDS=[41001,52001,63001,74001,85001]; LANDMARKS=[0,20,23,24]
elif MODE=='smoke':
    N=1000; EPOCHS=2; PATIENCE=2; M=64; N_EVAL=5
    SEEDS=[41001,52001]; LANDMARKS=[0,23,24]
else: raise ValueError('Choose full or smoke')
OUT=OUTPUT_PARENT/RUN_TAG
BATCH_SIZE=512
print('Output:',OUT,'M:',M)

## Fit or restore all 11 models
The settings below are explicit defaults, not tuned original-study configurations. DynamicDeepHit uses dynamic validation concordance for stopping; SurvTRACE uses validation loss. All training prefixes are used, with a fixed validation subset. Patients are split before prefix expansion. No test outcomes or analytical references are used for model selection.

Both SurvTRACE variants have the same architecture and auxiliary tasks. Their labels enforce the likelihood choice. For publication claims, tune on validation data with matched search budgets and repeat training seeds. Full DynamicDeepHit includes five auxiliary-only burn-in epochs before joint training.

In [ ]:
HISTORY_OPTIONS={
    'DynamicDeepHit': {
        'model_config': dict(hidden=64,layers=1,attention_hidden=64,head_hidden=64,head_layers=2,dropout=0.1),
        'loss_config': dict(rank_weight=0.1,rank_sigma=0.1,auxiliary_weight=0.1,l1_weight=1e-5,rank_reduction='per_patient'),
        'lr': 1e-3, 'batch_size': 256, 'monitor': 'concordance',
        'validation_prefix_limit': 2048, 'burn_in_epochs': 5 if MODE=='full' else 0,
    },
    'SurvTransformer': {
        'model_config': dict(width=16,heads=2,layers=3,intermediate=64,head_layers=2,dropout=0.1),
        'loss_config': dict(mortality_weight=1.,duration_weight=1.),
        'lr': 1e-3, 'batch_size': 256, 'weight_decay': 1e-4,
        'monitor': 'loss', 'validation_prefix_limit': 2048,
    },
    'SurvTransformer-Discrete': {
        'model_config': dict(width=16,heads=2,layers=3,intermediate=64,head_layers=2,dropout=0.1),
        'loss_config': dict(mortality_weight=1.,duration_weight=1.),
        'lr': 1e-3, 'batch_size': 256, 'weight_decay': 1e-4,
        'monitor': 'loss', 'validation_prefix_limit': 2048,
    },
}
experiment=train(OUT,n=N,epochs=EPOCHS,patience=PATIENCE,device=DEVICE,
                 history_options=HISTORY_OPTIONS,reuse_from=REUSE_FROM)
display(pd.DataFrame(experiment.config['training']).T)

## Evaluate against the same deterministic references
All seven Glazier methods and the true-generator IID sampler are retained. Direct models return curves without rollouts. New history models and Glazier see the same observed prefix. Lite, CPH, Weibull and DRSA still use baseline-survival updates.

The reference is the observed-generator population KM functional, with biological identification unresolved. Sampling error bars describe rollout-seed variability, not training uncertainty. Weighted branching and conditional Monte Carlo use live-particle budgets and weighted product limits. Sobol at 10,000 points lacks the strongest powers-of-two balance guarantees.

In [ ]:
replicates,summary=evaluate(experiment,OUT/'evaluation',M=M,n_eval=N_EVAL,
    seeds=SEEDS,landmarks=LANDMARKS,batch_size=BATCH_SIZE)
display(summary)


## Review errors, curves, runtime and training logs
MAE compares individual curves before averaging. Mean curves are descriptive and can hide patient-level error. Smoke scores are not performance evidence. Continuous-time and discrete-step SurvTRACE variants are separately labeled.

In [ ]:
plot(OUT/'evaluation')
display(summary.sort_values(['landmark','MAE']))
display(summary[['landmark','model','method','mean_seconds','min_ESS']])
for label in HISTORY_OPTIONS:
    log_path=OUT/'checkpoints'/(label+'_training.csv')
    if log_path.exists():
        print(label)
        display(pd.read_csv(log_path))

## Save all results
The archive includes cohorts, checkpoints, per-epoch training logs, configuration/source hashes, individual predictions, references, summary tables and plots. Keep the source bundle alongside it. New output names distinguish this comparison from earlier runs.

In [ ]:
import shutil
(OUT/'model_fidelity_notes.md').write_text((BUNDLE/'HISTORY_COMPARATORS.md').read_text(encoding='utf-8'),encoding='utf-8')
archive=Path(shutil.make_archive(str(OUT)+'_results','zip',root_dir=OUT))
print('Saved:',archive)
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))